# 03 One-hot 与 Embedding 矩阵

前两节已经得到 Token ID。这一节进入最核心的问题：一个整数 ID 到底怎样变成神经网络可以处理的向量。

## 1. 先回顾当前进度

目前已经完成：

$$
\text{原始文本}
\rightarrow
\text{Token 序列}
\rightarrow
\text{Token ID 序列}
$$

例如：

$$
\text{我喜欢深度学习}
\rightarrow
[\text{我},\text{喜欢},\text{深度},\text{学习}]
\rightarrow
[2,3,4,5]
$$

现在需要把每个 ID 转换成一个 $D$ 维向量。

## 2. One-hot 编码是什么

假设词表大小为 $V=6$。ID 为 2 的 token 可以表示成长度为 6 的 One-hot 向量：

$$
\mathbf{o}_2=
[0,0,1,0,0,0]
$$

这个向量只有 ID 对应的位置是 1，其余位置都是 0。

一般来说，ID 为 $i$ 的 One-hot 向量记作：

$$
\mathbf{o}_i \in \mathbb{R}^{V}
$$

One-hot 解决了“不要把 ID 当作具有大小关系的连续数值”这个问题。每个 token 都占据一个独立位置。

## 3. One-hot 的局限

One-hot 虽然区分了不同 token，但它不是理想的最终表示。

### 3.1 维度太大

如果词表有 50,000 个 token，每个 One-hot 向量就有 50,000 维，并且绝大多数位置都是 0。

### 3.2 不能表达相似性

任意两个不同 token 的 One-hot 向量互相正交。无论“猫”和“狗”在语义上多么相似，它们在 One-hot 表示中都没有比“猫”和“汽车”更接近。

因此还需要把稀疏的 One-hot 向量映射成较低维、可学习的稠密向量。

## 4. Embedding 矩阵

设词表大小为 $V$，希望每个 token 用 $D$ 个数表示，就建立一个 Embedding 矩阵：

$$
\mathbf{E}\in\mathbb{R}^{V\times D}
$$

矩阵有 $V$ 行，因为词表中有 $V$ 个 token；矩阵有 $D$ 列，因为每个 token 要得到一个 $D$ 维向量。

矩阵的第 $i$ 行，就是 ID 为 $i$ 的 token 所对应的向量：

$$
\mathbf{e}_i=\mathbf{E}_{i,:}\in\mathbb{R}^{D}
$$

## 5. 用一个小矩阵看懂查表

假设词表大小为 6，Embedding 维度为 3：

$$
\mathbf{E}=
\begin{bmatrix}
0.10 & 0.20 & -0.10 \\
-0.30 & 0.40 & 0.50 \\
0.70 & -0.20 & 0.10 \\
0.00 & 0.60 & -0.40 \\
0.20 & 0.30 & 0.80 \\
-0.50 & 0.10 & 0.40
\end{bmatrix}
\in\mathbb{R}^{6\times3}
$$

如果“我”的 Token ID 是 2，就取矩阵第 2 号行：

$$
\text{ID}=2
\quad\Longrightarrow\quad
\mathbf{E}_{2,:}=[0.70,-0.20,0.10]
$$

这里按照编程中的习惯从 0 开始编号，所以 ID 2 对应矩阵显示出来的第 3 行。

这就是 Embedding Lookup：Token ID 充当行号，模型取出对应行作为向量。

## 6. One-hot 乘矩阵为什么等于查表

ID 2 的 One-hot 向量是：

$$
\mathbf{o}_2=[0,0,1,0,0,0]
$$

它与 Embedding 矩阵相乘：

$$
\mathbf{o}_2\mathbf{E}
=
0\mathbf{E}_{0,:}
+0\mathbf{E}_{1,:}
+1\mathbf{E}_{2,:}
+0\mathbf{E}_{3,:}
+0\mathbf{E}_{4,:}
+0\mathbf{E}_{5,:}
$$

所有乘以 0 的行都会消失，只剩下：

$$
\mathbf{o}_2\mathbf{E}
=
\mathbf{E}_{2,:}
=
[0.70,-0.20,0.10]
$$

因此：

$$
\boxed{\text{One-hot}\times\text{Embedding 矩阵}=\text{按 ID 查出对应行}}
$$

数学讲解常使用 One-hot 乘法，实际程序通常直接查表，避免真的创建巨大而稀疏的 One-hot 向量。

## 7. 一整个序列怎样变成向量序列

假设一个句子的 Token ID 序列长度为 $L$：

$$
[i_1,i_2,\dots,i_L]
$$

对每个 ID 分别查表：

$$
[i_1,i_2,\dots,i_L]
\xrightarrow{\text{Embedding Lookup}}
\begin{bmatrix}
\mathbf{E}_{i_1,:} \\
\mathbf{E}_{i_2,:} \\
\vdots \\
\mathbf{E}_{i_L,:}
\end{bmatrix}
\in\mathbb{R}^{L\times D}
$$

原来每个位置只有一个整数，现在每个位置都拥有一个 $D$ 维向量。

## 8. Batch 中的形状变化

一次处理 $B$ 个句子，每个句子统一为 $L$ 个 token：

$$
\underbrace{B\times L}_{\text{Token ID}}
\xrightarrow{\text{Embedding}}
\underbrace{B\times L\times D}_{\text{Token 向量}}
$$

例如：

$$
32\times128
\rightarrow
32\times128\times768
$$

含义是：

- 一次处理 32 个句子。
- 每个句子有 128 个 token 位置。
- 每个 token 用 768 个数表示。

这就连接到了你在 Attention 笔记中见过的 $B\times N\times D$。这里只是把序列长度记作 $L$，含义相同。

## 9. Embedding 矩阵里的数从哪里来

Embedding 矩阵不是人工为每个词填写含义。

在一般的神经网络训练中，它是模型参数的一部分：开始时通常经过随机初始化，之后由损失函数、反向传播和优化器逐步更新。

$$
\text{Embedding 向量}
\rightarrow
\text{后续网络}
\rightarrow
\text{预测与损失}
\rightarrow
\text{反向传播更新 }\mathbf{E}
$$

你之前学习的梯度下降和反向传播同样作用于 Embedding 矩阵。这里只建立联系，具体训练过程留到后续课程展开。

## 10. 不要混淆初始 Embedding 与上下文表示

同一个 Token ID 每次查同一张 Embedding 矩阵时，会先得到相同的初始向量。

例如“苹果”出现在下面两句话中：

1. 我吃了一个苹果。
2. 苹果发布了新电脑。

如果 Tokenizer 得到的是同一个“苹果”token，那么 Embedding 查表得到的初始向量相同。它们经过 Transformer 并与不同上下文交互后，才会形成不同的上下文表示。

$$
\text{Token Embedding}
\rightarrow
\text{加入位置信息}
\rightarrow
\text{Transformer}
\rightarrow
\text{上下文表示}
$$

## 11. 三节课连起来看

$$
\boxed{
\text{文本}
\xrightarrow{\text{Tokenizer}}
\text{Token}
\xrightarrow{\text{词表}}
\text{Token ID}
\xrightarrow{\text{Embedding 查表}}
\text{向量}
}
$$

各部分的职责是：

- Tokenizer：决定文本怎样被切分。
- 词表：决定每个 token 使用哪个整数 ID。
- Embedding 矩阵：决定每个 ID 当前对应什么向量。
- Transformer：让初始向量进一步结合上下文信息。

## 12. 本节小结

1. One-hot 用独立位置表示 token，避免把 ID 的大小误当成语义大小。
2. One-hot 维度大而且稀疏，本身也不能表达 token 相似性。
3. Embedding 矩阵的形状是 $V\times D$，每一行对应一个 token 向量。
4. ID 为 $i$ 的 token 对应向量 $\mathbf{E}_{i,:}$。
5. One-hot 乘 Embedding 矩阵，在数学上等价于按照 ID 查出矩阵的一行。
6. Embedding 会把形状从 $B\times L$ 变成 $B\times L\times D$。
7. Embedding 矩阵通常是可训练参数，并通过反向传播更新。

## 13. 自测问题

1. 词表大小为 10,000 时，一个 One-hot 向量有多少维？
2. 为什么不同 token 的 One-hot 表示不能体现语义相似性？
3. 如果词表大小是 $V$、向量维度是 $D$，Embedding 矩阵是什么形状？
4. 为什么 One-hot 乘 Embedding 矩阵等价于查表？
5. 输入 Token ID 的形状是 $32\times128$，Embedding 维度是 768，输出形状是什么？
6. Token ID 是向量的内容，还是查找向量的地址？
7. Embedding 层输出和 Transformer 输出的上下文表示有什么区别？